# From Spreadsheet to SQL Database

## Activity overview

We have worked with data using both spreadsheets and SQL. These tools operate very differently: In spreadsheets, we are able to observe and interact with data directly; with SQL,we interact with data through queries to the database. In this notebook, we will use spreadsheets to clean our data before importing it into SQL for analysis. 

In this scenario, we have been working for a national store chain as a data analyst. Management is interested in the amount of inventory being kept in storage at regional sites. Our supervisor has asked us to perform an analysis on inventory and sales data to make recommendations for changes to inventory management practices. We have been provided with three datasets containing information about inventory, products, and sales. 

## Objective
Combine tools of spreadsheets and SQL to successfully analyze data.

## Step 1: Clean the data
We will use three CSV files: inventory, products, and sales, already downloaded on my local machine.

### inventory.csv File

Let's first navigate to the Inventory sheet and apply a filter to inspect the values.

In the StoreName column, we found a blank. 

We might be able to find what the missing value is and input it correctly using the filter. As I clear the Storename filter and use the StoreId column filter for other stores with the ID 21791. It appears that the other stores with this ID are all Dollar Tree, so it’s probably safe to input that as the StoreName value in the blank cell. 

### products.csv File

We repeat the filter process to inspect the Products data. 

We found that there is a NA value in the ProductID column, despite the fact that this column should only have numeric values. In this case, assume that we’ve checked in with the dataset owner, who said we can delete this row because it was input by mistake and does not belong in this dataset. So I delete this row. 

## Step 2: From Spreadsheets to SQL

Now that we have cleaned our data, it’s time to transition to using SQL. With SQL, we can only observe the results of our query, which requires a different mindset than spreadsheets — but SQL is very powerful when we’re working with databases and larger datasets!

Since we use local csv files, I use DuckDB for easier import. Let's name our table "sales_info","product_info",and "inventory_info".

### Create Dataset and Create Table

In [1]:
!pip install duckdb --trusted-host pypi.org --trusted-host files.pythonhosted.org

In [2]:
import duckdb

con = duckdb.connect("sales.duckdb")
con.sql("""
CREATE TABLE sales_info AS
SELECT *
FROM read_csv_auto('Sales.csv')
""")

### Inspect the Data

In [6]:
con.sql("""
SELECT *
FROM sales.sales_info
LIMIT 10;
""")

┌─────────┬─────────┬───────────┬────────────┬───────────────────┬──────────┐
│ SalesId │ StoreId │ ProductId │    Date    │     UnitPrice     │ Quantity │
│  int64  │  int64  │   int64   │    date    │      double       │  int64   │
├─────────┼─────────┼───────────┼────────────┼───────────────────┼──────────┤
│   82319 │   22726 │       590 │ 2019-12-02 │            0.0525 │       93 │
│   15022 │   21754 │       390 │ 2017-11-19 │ 5.109999999999999 │       28 │
│   11624 │   71053 │       883 │ 2020-07-13 │            7.3675 │       33 │
│   63101 │   22914 │       658 │ 2019-05-12 │            2.0825 │       76 │
│   29702 │   22623 │       632 │ 2020-07-20 │            0.6475 │        8 │
│   35660 │   22749 │       170 │ 2019-09-30 │              5.67 │       93 │
│   69913 │   22633 │       444 │ 2017-11-03 │            0.4375 │       98 │
│   47278 │   84969 │       184 │ 2018-04-17 │             9.205 │       17 │
│   46126 │   48187 │       316 │ 2017-10-13 │             3.395

#### Next, inspect the data to find out how many years of sales data it includes.

In [7]:
con.sql("""
SELECT
  MIN(Date) AS min_date,
  MAX(Date) AS max_date
FROM
    sales.sales_info;
""")

┌────────────┬────────────┐
│  min_date  │  max_date  │
│    date    │    date    │
├────────────┼────────────┤
│ 2017-01-01 │ 2020-12-30 │
└────────────┴────────────┘

Now we know what years this data covers. In this case, we’ll want to group the data by month because management wants to see year-over-year changes to inventory by month.

In [8]:
#return the total quantity sold for each ProductId grouped by the month and year it was sold: 

con.sql("""
SELECT
  EXTRACT(YEAR FROM date) AS Year,
  EXTRACT(MONTH FROM date) AS Month,
  ProductId,
  ROUND(MAX(UnitPrice),2) AS UnitPrice,
  SUM(Quantity) AS UnitsSold
FROM
  sales.sales_info
GROUP BY
  Year,
  Month,
  ProductId
ORDER BY
  Year,
  Month,
  ProductId;
""")

┌───────┬───────┬───────────┬───────────┬───────────┐
│ Year  │ Month │ ProductId │ UnitPrice │ UnitsSold │
│ int64 │ int64 │   int64   │  double   │  int128   │
├───────┼───────┼───────────┼───────────┼───────────┤
│  2017 │     1 │         2 │      5.23 │       231 │
│  2017 │     1 │         3 │       0.3 │       427 │
│  2017 │     1 │         4 │      9.24 │       159 │
│  2017 │     1 │         5 │      1.37 │       290 │
│  2017 │     1 │         6 │      0.65 │       362 │
│  2017 │     1 │         8 │      2.61 │        21 │
│  2017 │     1 │         9 │      4.08 │       488 │
│  2017 │     1 │        10 │      0.18 │       272 │
│  2017 │     1 │        11 │      1.54 │       232 │
│  2017 │     1 │        12 │      5.15 │       163 │
│    ·  │     · │         · │        ·  │         · │
│    ·  │     · │         · │        ·  │         · │
│    ·  │     · │         · │        ·  │         · │
│  2017 │    11 │       124 │       8.3 │        63 │
│  2017 │    11 │       125 

## Step 3: Export results to spreadsheet

The subset of data we queried is fewer than 50,000 rows. This means it can be easily exported to a spreadsheet, if our stakeholder requests the data in this form. Or, we can use this exported spreadsheet for visualization. 

In [9]:
con.sql("""
COPY (
    SELECT
        EXTRACT(YEAR FROM date) AS Year,
        EXTRACT(MONTH FROM date) AS Month,
        ProductId,
        ROUND(MAX(UnitPrice),2) AS UnitPrice,
        SUM(Quantity) AS UnitsSold
    FROM
        sales.sales_info
    GROUP BY
        Year,
        Month,
        ProductId
    ORDER BY
        Year,
        Month,
        ProductId
) TO 'sales_summary.csv' (HEADER, DELIMITER ',');
""")